# Property Price Predictor (ML)

A Tkinter-based Python desktop app that predicts Melbourne property sale prices using
**data preprocessing, PCA, and a Random Forest Regressor**, evaluated with the **R² score**.

This notebook documents the full pipeline that powers `app.py`:
1. Data loading & inspection
2. Preprocessing — drop non-predictive columns, impute missing values, encode categoricals, scale
3. Dimensionality reduction with PCA
4. Random Forest regression & R²/RMSE/MAE evaluation
5. Sample prediction

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from preprocessing import (
    DataPreprocessor, FrequencyEncoder, load_data, split_features_target, TARGET,
    DEFAULT_DROP_COLUMNS, LOW_CARD_CATS, HIGH_CARD_CATS,
)
from model import ModelTrainer, Metrics

plt.rcParams['figure.figsize'] = (8, 4)
CSV = 'Property Sales of Melbourne City.csv'

## 2. Load and inspect the data

In [ ]:
df = load_data(CSV)
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing.plot.barh(figsize=(7,5), color='steelblue')
plt.title('Missing values per column')
plt.show()

In [ ]:
df[TARGET].hist(bins=50, color='salmon', edgecolor='white')
plt.title('Price distribution')
plt.xlabel('Price (AUD)')
plt.show()

## 3. Preprocessing

**Columns dropped:** `Unnamed: 0` (row index) and `Address` (18,134 unique strings — no generalisable signal for a learned model). This is the *Address column drop* step.

**Missing values:** numeric → median imputation, categorical → mode imputation.

**Encoding:**
- Low-cardinality categoricals (`Type`, `Method`, `Regionname`, `CouncilArea`) → one-hot encoding.
- High-cardinality categoricals (`Suburb` with 330 levels, `SellerG` with 305) → custom `FrequencyEncoder` to keep the feature space compact.

**Scaling:** `StandardScaler` before PCA.

**Date:** the `Date` column is expanded into `SaleYear` / `SaleMonth`.

In [ ]:
print('Default drop columns:', DEFAULT_DROP_COLUMNS)
print('One-hot encoded:', LOW_CARD_CATS)
print('Frequency encoded:', HIGH_CARD_CATS)
print('Address unique values:', df['Address'].nunique())

In [ ]:
X, y = split_features_target(df, TARGET)
print('Features:', X.shape[1], '| Target:', TARGET)
X.head()

### 3.1 Build and fit the preprocessing pipeline

In [ ]:
preprocessor = DataPreprocessor(
    drop_columns=['Unnamed: 0', 'Address'],
    numeric_impute_strategy='median',
    categorical_impute_strategy='most_frequent',
    scale=True,
    use_pca=True,
    pca_variance=0.95,
)
Xt = preprocessor.fit_transform(X)
print('Raw feature columns:', X.shape[1])
print('Transformed shape:', Xt.shape)
print('PCA components retained:', preprocessor.pca_.n_components_)

### 3.2 PCA explained variance

In [ ]:
ev = preprocessor.explained_variance_ratio_
cum = np.cumsum(ev)
plt.bar(range(1, len(ev)+1), ev, alpha=0.7, label='individual')
plt.plot(range(1, len(ev)+1), cum, color='crimson', marker='o', label='cumulative')
plt.axhline(0.95, color='gray', linestyle='--')
plt.xlabel('Principal component')
plt.ylabel('Variance ratio')
plt.title('PCA explained variance')
plt.legend()
plt.show()

## 4. Random Forest regression

The preprocessing pipeline and a `RandomForestRegressor` are composed into a single scikit-learn `Pipeline` so the whole model can be fitted, evaluated, and serialised as one unit.

In [ ]:
trainer = ModelTrainer(
    preprocessor=preprocessor,
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    max_features='sqrt',
    random_state=42,
)
result = trainer.fit(df, test_size=0.2)
result.metrics.as_dict()

### 4.1 Evaluation metrics

In [ ]:
m = result.metrics
print(f'R²  (train): {m.r2_train:.4f}')
print(f'R²  (test) : {m.r2_test:.4f}')
print(f'RMSE(train) : {m.rmse_train:,.0f}')
print(f'RMSE(test)  : {m.rmse_test:,.0f}')
print(f'MAE (train) : {m.mae_train:,.0f}')
print(f'MAE (test)  : {m.mae_test:,.0f}')

### 4.2 Component importances

In [ ]:
imp = result.importances
labels = result.component_labels
order = np.argsort(imp)[::-1][:15]
plt.barh([labels[i] for i in order][::-1], imp[order][::-1], color='steelblue')
plt.title('Top 15 PCA-component importances')
plt.show()

## 5. Sample prediction

In [ ]:
sample = X.iloc[0:1].to_dict(orient='records')[0]
predicted = trainer.predict_one(sample)
actual = float(y.iloc[0])
print(f'Predicted: A$ {predicted:,.0f}')
print(f'Actual   : A$ {actual:,.0f}')
print(f'Error    : {abs(predicted-actual)/actual:.1%}')

## 6. Save the trained model

In [ ]:
trainer.save('price_predictor.joblib')
print('Saved → price_predictor.joblib')

## 7. Launch the interactive desktop app

The Tkinter GUI in `app.py` wraps this whole pipeline with tabs for Overview, Preprocess, Train, Predict, and Insights:

```bash
python app.py
```